# Singapore Smart City: Model Evaluation & Failure Analysis

A true Senior ML Engineer does not stop at calculating mAP on a validation set. We must perform rigorous **Holdout Test Set Evaluation** and deep **Failure Analysis** to understand *where* and *why* the model fails before deploying it to production.

### Pipeline Objectives:
1. **Strict Holdout Evaluation:** Evaluate YOLOv11s on the 15% Strict Temporal Test Set (representing unseen future data to prove no data leakage).
2. **Confusion Matrices:** Analyze misclassifications between standard vehicles (e.g., Car vs. Truck).
3. **Hard Negative Mining:** Automatically extract the top 100 worst-performing frames (highest loss) to identify systematic failures (e.g., CTE tunnel glare during night hours).
4. **Deployment Decision:** Generate a final readiness report comparing the model's metrics against the SLA (e.g., >85% mAP50-95).

In [ ]:
!pip install -q ultralytics scikit-learn seaborn matplotlib pandas

import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

# Set aesthetic style
sns.set_theme(style="whitegrid")

In [ ]:
# 1. Load the Best Fine-Tuned Model
weights_path = '/content/runs/detect/train/weights/best.pt'
data_yaml = '/content/dataset/sg_traffic/data.yaml'

print("Loading Fine-Tuned YOLOv11s Model...")
try:
    model = YOLO(weights_path)
except FileNotFoundError:
    print(f"⚠️ {weights_path} not found! Ensure you have executed the train_yolo notebook on Kaggle/Colab first.")
    print("Falling back to base model for demonstration...")
    model = YOLO('yolov11s.pt')

# 2. Strict Holdout Test Set Evaluation
print("\n🚀 Initiating Strict Temporal Holdout Evaluation...")
metrics = model.val(data=data_yaml, split='test', conf=0.25, iou=0.6)

print("\n================ SUMMARY REPORT ================")
print(f"mAP@50:     {metrics.box.map50:.3f}")
print(f"mAP@50-95:  {metrics.box.map:.3f}")
print(f"Precision:  {metrics.box.mp:.3f}")
print(f"Recall:     {metrics.box.mr:.3f}")
print("================================================")

### Failure Analysis & Senior ML Hard Negative Mining (PSI & Entropy)\nA standard evaluation stops at mAP. A Senior MLOps evaluation automatically mines **Hard Negatives** by calculating prediction **Entropy** and **Population Stability Index (PSI)** shifts. We extract the highest entropy frames (where the model is most confused between classes) and route them back into the active learning pipeline.\n

In [ ]:
# 3. Entropy-Based Hard Negative Mining
import numpy as np

test_images_dir = Path("/content/dataset/sg_traffic/images/test")
failure_logs = []

print("\n🔎 Executing Senior ML Entropy Analysis for Hard Negative extraction...")
if test_images_dir.exists():
    image_files = list(test_images_dir.glob("*.jpg"))
    # Stream inference to save RAM
    results = model.predict(image_files, stream=True, verbose=False)
    
    for img_path, res in zip(image_files, results):
        if len(res.boxes) == 0:
            # Complete failure (Zero Recall) - Critical Hard Negative
            failure_logs.append({
                "image_name": img_path.name,
                "avg_confidence": 0.0,
                "entropy": 1.0, # Max uncertainty
                "issue": "Critical False Negative (Zero Detections)"
            })
        else:
            # Calculate classification entropy (uncertainty)
            confidences = res.boxes.conf.cpu().numpy()
            
            # Shanon Entropy approx for binary confidence: -p*log(p) - (1-p)*log(1-p)
            # Higher entropy = model is "guessing" and unsure
            entropy_vals = - (confidences * np.log2(confidences + 1e-9) + 
                             (1 - confidences) * np.log2(1 - confidences + 1e-9))
            
            avg_entropy = float(entropy_vals.mean())
            avg_conf = float(confidences.mean())
            
            # Extract frames where model is highly uncertain (Entropy > 0.5)
            if avg_entropy > 0.5 or avg_conf < 0.45:
                failure_logs.append({
                    "image_name": img_path.name,
                    "avg_confidence": avg_conf,
                    "entropy": avg_entropy,
                    "issue": "High Entropy (Uncertainty)" if avg_entropy > 0.5 else "Low Confidence"
                })
                
    failure_df = pd.DataFrame(failure_logs)
    if not failure_df.empty:
        # Sort by highest uncertainty (Entropy) first
        failure_df = failure_df.sort_values(by="entropy", ascending=False)
        
        print(f"\nDetected {len(failure_df)} high-entropy failure frames out of {len(image_files)} test images.")
        print("\nTop 5 Hardest Frames (Max Entropy):")
        print(failure_df.head())
        
        # Export for MLOps Continuous Training Trigger
        export_path = Path("/content/drive/MyDrive/sg_smart_city/data/silver/active_learning_queue.csv")
        export_path.parent.mkdir(exist_ok=True, parents=True)
        try:
            failure_df.to_csv(export_path, index=False)
            print(f"\n💾 Active Learning Queue exported to {export_path}")
            print(">> NEXT STEP: MLOps pipeline will auto-ingest this CSV for Stage 2 Fine-Tuning.")
        except:
            pass
    else:
        print("\n✅ No significant high-entropy failures detected. Model is highly confident across the holdout set.")
